# WBF 앙상블 제출 노트북

Drive에 저장된 여러 체크포인트를 WBF로 합산해 Kaggle에 제출합니다.

In [ ]:
import os

# ══════════════════════════════════════════════════════════════════
# ① 여기만 수정하세요
# ══════════════════════════════════════════════════════════════════

REPO_URL       = "https://github.com/beomjinkim2000/Code_IT_Team_1_FirstProject.git"
BRANCH         = "main"
PROJECT_DIR    = "/content/Code_IT_Team_1_FirstProject"
COMPETITION_ID = "ai11-level1-project"

SUBMIT_TITLE   = "WBF ensemble v1"
SUBMIT_DESC    = "all checkpoints WBF"

WBF_CHECKPOINTS = [
    "/content/drive/MyDrive/HealthEat/checkpoints/best_model(V4-1).pt",
    "/content/drive/MyDrive/HealthEat/checkpoints/best_model(V4-2).pt",
    "/content/drive/MyDrive/HealthEat/checkpoints/best_model(V4-3).pt",
    "/content/drive/MyDrive/HealthEat/checkpoints/best_model(V2.7.6).pt",
    "/content/drive/MyDrive/HealthEat/checkpoints/best_model(V2.7.5).pt",
    "/content/drive/MyDrive/HealthEat/checkpoints/best_model(V2.7.3).pt",
    "/content/drive/MyDrive/HealthEat/checkpoints/하이퍼_best_model.pt",
    "/content/drive/MyDrive/HealthEat/checkpoints/창준님_에폭40_best_model.pt",
]
WBF_USE_EMA      = [True, True, True, True, True, True, True, True]
WBF_WEIGHTS      = [2, 1, 2, 2, 2, 1, 2, 2]
WBF_IOU_THR      = 0.5
WBF_SKIP_BOX_THR = 0.25

DRIVE_SUB_DIR = "/content/drive/MyDrive/HealthEat/submissions"
# ══════════════════════════════════════════════════════════════════

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} checkout {BRANCH}
    !git -C {PROJECT_DIR} pull origin {BRANCH}

os.chdir(PROJECT_DIR)
print(f"작업 디렉토리: {os.getcwd()}")

## 1. 패키지 설치

In [ ]:
!pip install -q \
    ultralytics \
    torchmetrics[detection] \
    albumentations \
    kaggle \
    pyyaml \
    tqdm \
    iterative-stratification \
    ensemble-boxes

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Kaggle 인증 + 데이터 다운로드

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

raw_data_path = Path("data/raw/ai11-level1-project")
data_dir      = raw_data_path / "sprint_ai_project1_data"
raw_data_path.mkdir(parents=True, exist_ok=True)

if data_dir.exists():
    print("데이터 이미 존재 — 스킵")
else:
    !kaggle competitions download -c {COMPETITION_ID} -p {raw_data_path}
    for zf in raw_data_path.glob("*.zip"):
        with zipfile.ZipFile(zf) as z:
            z.extractall(raw_data_path)
        zf.unlink()
    print("압축 해제 완료")

## 3. Google Drive 마운트 + 체크포인트 복원

In [ ]:
from google.colab import drive
import shutil, time
from pathlib import Path

drive.mount("/content/drive")

ckpt_dir = Path("outputs/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)

def copy_with_retry(src, dst, retries=3):
    for attempt in range(retries):
        try:
            shutil.copy(src, dst)
            return
        except OSError as e:
            if attempt < retries - 1:
                print(f"  Drive 연결 끊김 — 재마운트 후 재시도 ({attempt+1}/{retries-1})")
                drive.flush_and_unmount()
                time.sleep(2)
                drive.mount("/content/drive", force_remount=True)
            else:
                raise

local_ckpts = []
for i, drive_ckpt in enumerate(WBF_CHECKPOINTS):
    src = Path(drive_ckpt)
    if not src.exists():
        raise FileNotFoundError(f"Drive에 파일 없음: {src}")
    dst = ckpt_dir / f"ensemble_model_{i}.pt"
    copy_with_retry(src, dst)
    local_ckpts.append(dst)
    print(f"[{i}] {src.name} → {dst.name} 복사 완료")

## 4. 각 체크포인트 예측

In [ ]:
from pathlib import Path
import shutil

pred_dir = Path("outputs/predictions")
sub_dir  = Path("outputs/submissions")
pred_dir.mkdir(parents=True, exist_ok=True)
sub_dir.mkdir(parents=True, exist_ok=True)

# WBF_USE_EMA / WBF_WEIGHTS가 체크포인트보다 짧으면 마지막 값으로 채움
n = len(local_ckpts)
use_ema_list = (WBF_USE_EMA + [WBF_USE_EMA[-1]] * n)[:n]
weights_list = (WBF_WEIGHTS + [WBF_WEIGHTS[-1]] * n)[:n]

pred_json_paths = []
for i, (local_ckpt, use_ema) in enumerate(zip(local_ckpts, use_ema_list)):
    ema_flag = "--use-ema" if use_ema else ""
    print(f"\n[{i}] 예측 시작: {local_ckpt.name}  (EMA={use_ema})")
    !python predict.py --config configs/default.yaml --checkpoint {local_ckpt} {ema_flag}

    pred_json = pred_dir / f"pred_ensemble_{i}.json"
    shutil.copy(pred_dir / "predictions.json", pred_json)
    pred_json_paths.append(str(pred_json))
    print(f"[{i}] 예측 완료 → {pred_json.name}")

WBF_WEIGHTS = weights_list  # 실제 사용 가중치로 갱신

## 5. WBF 앙상블 → submission.csv 생성

In [ ]:
from pathlib import Path
import shutil, subprocess

drive_sub = Path(DRIVE_SUB_DIR)
drive_sub.mkdir(parents=True, exist_ok=True)

wbf_csv = sub_dir / "submission_wbf.csv"

if len(pred_json_paths) >= 2:
    weights_str = " ".join(str(w) for w in WBF_WEIGHTS)
    preds_str   = " ".join(pred_json_paths)
    !python ensemble_wbf.py \
        --preds {preds_str} \
        --weights {weights_str} \
        --iou-thr {WBF_IOU_THR} \
        --skip-box-thr {WBF_SKIP_BOX_THR} \
        --output {wbf_csv}
    print("WBF 앙상블 완료")
else:
    # 단일 모델 — predict.py가 만든 submission_v1.csv를 그대로 사용
    single_csv = sub_dir / "submission_v1.csv"
    shutil.copy(single_csv, wbf_csv)
    print("모델 1개 → WBF 생략, submission_v1.csv 복사")

shutil.copy(wbf_csv, drive_sub / wbf_csv.name)
print(f"\n완료: {wbf_csv}")
print(f"Drive 저장: {drive_sub / wbf_csv.name}")

## 6. Kaggle 제출

In [ ]:
!kaggle competitions submit \
    -c {COMPETITION_ID} \
    -f {wbf_csv} \
    -m "{SUBMIT_TITLE}: {SUBMIT_DESC}"

print(f"\n제출 완료!")
print(f"  제목: {SUBMIT_TITLE}")
print(f"  설명: {SUBMIT_DESC}")


## 7. 제출 내역 확인

In [ ]:
!kaggle competitions submissions -c {COMPETITION_ID}
